# GPU Final Train Notebook - Units SAEs

This notebook retrains unit-specific SAEs for all selected layers (4, 8, 12, 16, 20, 24, 28), saves them in a separate `final_train` checkpoint folder, rebuilds the unit contrast graphs, and reruns only the affected unit interventions.

The previous Drive checkpoints remain untouched. This run is meant to test whether the weak sparse SAE interventions were caused by using mixed/general SAEs rather than unit-specialized SAEs across the selected layers.


## Step 1 - Mount Drive and fetch code

GitHub is preferred for code. Drive `project.zip` remains a fallback. SAE checkpoints and tuned outputs persist in Drive.


In [2]:
# Step 1: Mount Google Drive and fetch project code
# GitHub is the primary code source. Drive project.zip remains a backup.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess

repo_url = "https://github.com/evey-dev/test_run.git"
repo_dir = "/content/test_run"

def run_cmd(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

github_ok = False
try:
    clone_dir = repo_dir
    if os.path.isdir(os.path.join(clone_dir, ".git")):
        run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
    else:
        if os.path.exists(clone_dir) and os.listdir(clone_dir):
            clone_dir = "/content/test_run_github"
        if os.path.isdir(os.path.join(clone_dir, ".git")):
            run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
        elif os.path.exists(clone_dir) and os.listdir(clone_dir):
            raise RuntimeError(f"{clone_dir} exists but is not a git repo")
        else:
            run_cmd(["git", "clone", "--depth", "1", repo_url, clone_dir])
    os.chdir(clone_dir)
    github_ok = True
    print(f"Using GitHub checkout: {os.getcwd()}")
except Exception as exc:
    print("GitHub checkout failed; falling back to Drive project.zip.")
    print(repr(exc))

if not github_ok:
    zip_path = "/content/drive/MyDrive/mphil-project/project.zip"
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Could not find {zip_path}. Fix GitHub access or upload project.zip.")
    print(f"Extracting backup zip {zip_path}...")
    run_cmd(["unzip", "-q", "-o", zip_path, "-d", "/content/"])
    for candidate in ["/content/test_run", "/content/mphil_project/test_run", "/content"]:
        if os.path.isdir(os.path.join(candidate, "src")) and os.path.isdir(os.path.join(candidate, "configs")):
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError("Could not find extracted project root containing src/ and configs/.")
    print(f"Using Drive backup project: {os.getcwd()}")

print(f"Current working directory: {os.getcwd()}")
!ls -l


Mounted at /content/drive
$ git -C /content/test_run pull --ff-only
Using GitHub checkout: /content/test_run
Current working directory: /content/test_run
total 2236
-rw-r--r-- 1 root root  10017 Jul 11 06:54 ai_handoff_summary.txt
-rw-r--r-- 1 root root   8944 Jul 11 06:54 changes_summary.txt
drwxr-xr-x 2 root root   4096 Jul 11 06:54 configs
drwxr-xr-x 2 root root   4096 Jul 11 06:54 data
-rw-r--r-- 1 root root    264 Jul 11 06:54 environment.yml
-rw-r--r-- 1 root root  19186 Jul 11 06:54 IMPLEMENTATION_PLAN.md
-rw-r--r-- 1 root root   1579 Jul 11 06:54 Instructions.md
drwxr-xr-x 3 root root   4096 Jul 11 06:54 mechanistic_data
drwxr-xr-x 5 root root   4096 Jul 11 06:54 outputs
-rw-r--r-- 1 root root   6551 Jul 11 06:54 project.txt
-rw-r--r-- 1 root root  14222 Jul 11 06:54 README.md
drwxr-xr-x 3 root root   4096 Jul 11 06:54 report
-rw-r--r-- 1 root root  17649 Jul 11 06:54 report_notes.txt
-rw-r--r-- 1 root root 158166 Jul 11 06:54 run_gpu_capitals_final.ipynb
-rw-r--r-- 1 root root

## Step 2 - Install dependencies


In [3]:
# Step 2: Install dependencies
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 156.5 MB/s eta 0:00:0000:010:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1
Obtaining file:///content/test_run
  Preparing metadata (setup.py) ... done
  Running setup.py develop for mechanistic-interpretability-repro


## Step 3 - Generate units data


In [4]:
# Step 3: Generate datasets
!python data/generate_datasets.py
print("Dataset files:")
!ls -lh data/units_data.csv


Generating Datasets...

Skipping capital sentence generation. (Pass --capitals flag if needed).
Standalone Mode complete:
Saved 1000 addition problems and 
Saved 1000 unit problems.
Dataset files:
-rw-r--r-- 1 root root 151K Jul 11 06:58 data/units_data.csv


## Step 4 - Define final-run folders

The final checkpoints use a new Drive folder, so this will not overwrite or silently resume from the earlier `sae_checkpoints_units_l24_l28_hyper` run.


In [6]:
# Final run folders
from pathlib import Path

FINAL_LAYERS = [4, 8, 12, 16, 20, 24, 28]
FINAL_DATA_DIR = 'mechanistic_data_units_final_train'
FINAL_SAE_DIR = 'mechanistic_data/sae_checkpoints_units_final_train'
FINAL_DRIVE_SAE_DIR = '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_units_final_train'
FINAL_CONFIG = 'configs/sae_units_final_train_config.yaml'

Path('configs').mkdir(exist_ok=True)
print('Final layers:', FINAL_LAYERS)
print('Activation dir:', FINAL_DATA_DIR)
print('Local SAE dir:', FINAL_SAE_DIR)
print('Drive SAE dir:', FINAL_DRIVE_SAE_DIR)
print('Config:', FINAL_CONFIG)


Final layers: [4, 8, 12, 16, 20, 24, 28]
Activation dir: mechanistic_data_units_final_train
Local SAE dir: mechanistic_data/sae_checkpoints_units_final_train
Drive SAE dir: /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_units_final_train
Config: configs/sae_units_final_train_config.yaml


## Step 5 - Capture units-only activations for all selected layers


In [9]:
# Capture units-only MLP activations for final SAE training
!python src/capture_activations.py \
  --output-dir mechanistic_data_units_final_train \
  --behaviours units \
  --layers 4 8 12 16 20 24 28 \
  --seed 787

print("\nFinal units-only activation bundle:")
!ls -lh mechanistic_data_units_final_train


Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 185.84it/s]
Loading prompts for behaviour: units
Total prompts loaded: 1000
Running forward passes to capture activations...
100% 1000/1000 [00:55<00:00, 18.04it/s]
Saving activation tensors to disk...
Saved activations for layer 4: (1000, 2560) to /content/test_run/mechanistic_data_units_final_train/activations_layer4.npy
Saved activations for layer 8: (1000, 2560) to /content/test_run/mechanistic_data_units_final_train/activations_layer8.npy
Saved activations for layer 12: (1000, 2560) to /content/test_run/mechanistic_data_units_final_train/activations_layer12.npy
Saved activations for layer 16: (1000, 2560) to /content/test_run/mechanistic_data_units_final_train/activations_layer16.npy
Saved activations for layer 20: (1000, 2560) to /content/test_run/mechanistic_data_units_final_train/activat

## Step 6 - Train final unit-specific SAEs for all selected layers


In [7]:
# Train final unit-specific SAEs for all selected layers
# The config is written in the runtime too, so this notebook works even before a GitHub push.
from pathlib import Path

# Path('configs/sae_units_final_train_config.yaml').write_text('''output_dir: "mechanistic_data/sae_checkpoints_units_final_train"
# data_dir: "mechanistic_data_units_final_train"
# layers:
#   - 4
#   - 8
#   - 12
#   - 16
#   - 20
#   - 24
#   - 28
# hidden_size: 2560
# latent_dim: 8192
# batch_size: 64
# epochs: 75
# lr: 0.001
# l1_lambda: 0.0003
# seed: 787
# device: "auto"
# ''')
# print('Wrote configs/sae_units_final_train_config.yaml')

!python src/train.py \
  --config configs/sae_units_final_train_config.yaml \
  --drive-dir /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_units_final_train

print("\nFinal local checkpoints:")
!ls -lh mechanistic_data/sae_checkpoints_units_final_train
print("\nFinal Drive checkpoints:")
!ls -lh /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_units_final_train


Using CUDA device: NVIDIA A100-SXM4-80GB

=== Training SAE for layer 4 ===
Layer 4 activation scaling factor (raw mean L2 / sqrt(d_in)): 0.1605
Epoch 00 | train_loss=0.1445 | val_loss=0.0644 | mse=0.0643 | l1=0.1901
Epoch 01 | train_loss=0.0420 | val_loss=0.0228 | mse=0.0227 | l1=0.1836
Epoch 02 | train_loss=0.0169 | val_loss=0.0110 | mse=0.0109 | l1=0.1563
Epoch 03 | train_loss=0.0095 | val_loss=0.0093 | mse=0.0092 | l1=0.1374
Epoch 04 | train_loss=0.0063 | val_loss=0.0052 | mse=0.0052 | l1=0.1261
Epoch 05 | train_loss=0.0048 | val_loss=0.0043 | mse=0.0043 | l1=0.1203
Epoch 06 | train_loss=0.0036 | val_loss=0.0038 | mse=0.0038 | l1=0.1174
Epoch 07 | train_loss=0.0032 | val_loss=0.0035 | mse=0.0034 | l1=0.1154
Epoch 08 | train_loss=0.0027 | val_loss=0.0029 | mse=0.0028 | l1=0.1137
Epoch 09 | train_loss=0.0024 | val_loss=0.0026 | mse=0.0026 | l1=0.1121
Epoch 10 | train_loss=0.0022 | val_loss=0.0026 | mse=0.0025 | l1=0.1110
Epoch 11 | train_loss=0.0020 | val_loss=0.0023 | mse=0.0023 | l1

## Step 7 - Verify final checkpoint set


In [8]:
# Verify final checkpoint directory before running expensive attribution graphs
from pathlib import Path

layers = [4, 8, 12, 16, 20, 24, 28]
ckpt_dir = Path('mechanistic_data/sae_checkpoints_units_final_train')
missing = []
for layer in layers:
    for name in [f'sae_layer{layer}.pt', f'sae_layer{layer}_metadata.json', f'latents_layer{layer}.npy']:
        if not (ckpt_dir / name).exists():
            missing.append(str(ckpt_dir / name))

if missing:
    raise FileNotFoundError('Missing final SAE artifacts:\n' + '\n'.join(missing))

print('All final SAE artifacts found.')
print('Final SAE config:')
!cat configs/sae_units_final_train_config.yaml
print('Checkpoint directory:')
!ls -lh mechanistic_data/sae_checkpoints_units_final_train


All final SAE artifacts found.
Final SAE config:
output_dir: "mechanistic_data/sae_checkpoints_units_final_train"
data_dir: "mechanistic_data_units_final_train"
layers:
  - 4
  - 8
  - 12
  - 16
  - 20
  - 24
  - 28
hidden_size: 2560
latent_dim: 8192
batch_size: 64
epochs: 75
lr: 0.001
l1_lambda: 0.0003
seed: 787
device: "auto"
Checkpoint directory:
total 1.4G
-rw-r--r-- 1 root root  32M Jul 10 02:57 latents_layer12.npy
-rw-r--r-- 1 root root  32M Jul 10 02:57 latents_layer16.npy
-rw-r--r-- 1 root root  32M Jul 10 02:57 latents_layer20.npy
-rw-r--r-- 1 root root  32M Jul 10 02:57 latents_layer24.npy
-rw-r--r-- 1 root root  32M Jul 10 02:57 latents_layer28.npy
-rw-r--r-- 1 root root  32M Jul 10 02:56 latents_layer4.npy
-rw-r--r-- 1 root root  32M Jul 10 02:57 latents_layer8.npy
-rw-r--r-- 1 root root  15K Jul 10 02:57 sae_layer12_metadata.json
-rw-r--r-- 1 root root 161M Jul 10 02:57 sae_layer12.pt
-rw-r--r-- 1 root root  15K Jul 10 02:57 sae_layer16_metadata.json
-rw-r--r-- 1 root root

In [9]:
# Persist final SAE checkpoints to Drive in a clearly named finals folder
import os, shutil, glob
from pathlib import Path

local_final_sae = Path('/content/test_run/mechanistic_data/sae_checkpoints_units_final_train')
drive_final_sae = Path('/content/drive/MyDrive/mphil-project/mechanistic_data/finals/sae_checkpoints_units_final_train')

drive_final_sae.mkdir(parents=True, exist_ok=True)

required_layers = [4, 8, 12, 16, 20, 24, 28]
missing = []
for layer in required_layers:
    for name in [
        f'sae_layer{layer}.pt',
        f'latents_layer{layer}.npy',
        f'sae_layer{layer}_metadata.json',
    ]:
        src = local_final_sae / name
        if not src.exists():
            missing.append(str(src))

if missing:
    raise FileNotFoundError("Missing final SAE artifacts:\n" + "\n".join(missing))

for src in sorted(local_final_sae.glob('*')):
    if src.is_file():
        dst = drive_final_sae / src.name
        shutil.copy2(src, dst)
        print(f'Copied {src.name}')

print(f'\nSaved final checkpoints to: {drive_final_sae}')
!du -sh {drive_final_sae}

Copied latents_layer12.npy
Copied latents_layer16.npy
Copied latents_layer20.npy
Copied latents_layer24.npy
Copied latents_layer28.npy
Copied latents_layer4.npy
Copied latents_layer8.npy
Copied sae_layer12.pt
Copied sae_layer12_metadata.json
Copied sae_layer16.pt
Copied sae_layer16_metadata.json
Copied sae_layer20.pt
Copied sae_layer20_metadata.json
Copied sae_layer24.pt
Copied sae_layer24_metadata.json
Copied sae_layer28.pt
Copied sae_layer28_metadata.json
Copied sae_layer4.pt
Copied sae_layer4_metadata.json
Copied sae_layer8.pt
Copied sae_layer8_metadata.json

Saved final checkpoints to: /content/drive/MyDrive/mphil-project/mechanistic_data/finals/sae_checkpoints_units_final_train
1.4G	/content/drive/MyDrive/mphil-project/mechanistic_data/finals/sae_checkpoints_units_final_train


## Step 8 - Rebuild units contrast graphs with final unit-specific SAEs


In [10]:
# Final graph 1: force -> newtons, contrast joules
!python src/attribution_graph.py \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --target "newtons" \
  --contrast-target "joules" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_final_train_config.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/units_final_force_graph.json \
  --output-html outputs/units_final_force_graph.html \
  --output-mermaid outputs/units_final_force_graph.md


Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 192.79it/s]
Disabling gradients for all model parameters for speed and memory efficiency.
Loading SAE models from /content/test_run/mechanistic_data/sae_checkpoints_units_final_train to device: cuda...
Loaded SAE for layer 4 (scaling factor: 0.1605)
Loaded SAE for layer 8 (scaling factor: 0.2888)
Loaded SAE for layer 12 (scaling factor: 0.3036)
Loaded SAE for layer 16 (scaling factor: 0.2701)
Loaded SAE for layer 20 (scaling factor: 0.3199)
Loaded SAE for layer 24 (scaling factor: 0.6417)
Loaded SAE for layer 28 (scaling factor: 1.2420)
Running forward pass...

Top 5 model predictions:
  'new' (prob: 0.8164, logit: 20.62)
  'Newton' (prob: 0.1108, logit: 18.62)
  'p' (prob: 0.0280, logit: 17.25)
  'New' (prob: 0.0150, logit: 16.62)
  'N' (prob: 0.0150, logit: 16.62)

Target token for attribution

In [11]:
# Final graph 2: energy -> joules, contrast newtons
!python src/attribution_graph.py \
  --prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --target "joules" \
  --contrast-target "newtons" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_final_train_config.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/units_final_energy_graph.json \
  --output-html outputs/units_final_energy_graph.html \
  --output-mermaid outputs/units_final_energy_graph.md


Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 183.77it/s]
Disabling gradients for all model parameters for speed and memory efficiency.
Loading SAE models from /content/test_run/mechanistic_data/sae_checkpoints_units_final_train to device: cuda...
Loaded SAE for layer 4 (scaling factor: 0.1605)
Loaded SAE for layer 8 (scaling factor: 0.2888)
Loaded SAE for layer 12 (scaling factor: 0.3036)
Loaded SAE for layer 16 (scaling factor: 0.2701)
Loaded SAE for layer 20 (scaling factor: 0.3199)
Loaded SAE for layer 24 (scaling factor: 0.6417)
Loaded SAE for layer 28 (scaling factor: 1.2420)
Running forward pass...

Top 5 model predictions:
  'j' (prob: 0.7969, logit: 19.88)
  'J' (prob: 0.1226, logit: 18.00)
  'new' (prob: 0.0240, logit: 16.38)
  'the' (prob: 0.0101, logit: 15.50)
  'Newton' (prob: 0.0095, logit: 15.44)

Target token for attribution

## Step 9 - Rerun affected units interventions with final unit-specific SAEs


In [12]:
# Final full latent swap: ENERGY -> FORCE
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_final_train_config.yaml \
  --target-token "newtons, joules, volts" \
  --positions all \
  --layer-scan \
  --output outputs/units_final_swap_minpair_energy_to_force.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 188.13it/s]

[1/3] Clean Model Baseline:
  - Top prediction: 'new' (prob: 0.8164, logit: 20.62)
  - Target 'j': prob: 0.0000, logit: 10.06
  - Target 'J': prob: 0.0000, logit: 9.94
  - Target ' jou': prob: 0.0000, logit: 5.34
  - Target 'new': prob: 0.8164, logit: 20.62
  - Target ' new': prob: 0.0001, logit: 12.00
  - Target 'Newton': prob: 0.1108, logit: 18.62
  - Target 'p': prob: 0.0280, logit: 17.25
  - Target 'vol': prob: 0.0000, logit: 7.09
  - Target ' volts': prob: 0.0000, logit: -0.35

Source baseline prediction on 'Fact: The official SI unit for the energy of a moving engine thrust is named "':
  - Target 'j': prob: 0.7969, logit: 19.88
  - Target 'J': prob: 0.1226, logit: 18.00
  - Target ' jou': prob: 0.0005, logit: 12.56
  - Target 'new': prob: 0.0240, logit: 16.38
  - Target ' new': prob: 0.000

In [13]:
# Final full latent swap: FORCE -> ENERGY
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_final_train_config.yaml \
  --target-token "newtons, joules, volts" \
  --positions all \
  --layer-scan \
  --output outputs/units_final_swap_minpair_force_to_energy.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 191.65it/s]

[1/3] Clean Model Baseline:
  - Top prediction: 'j' (prob: 0.7969, logit: 19.88)
  - Target 'J': prob: 0.1226, logit: 18.00
  - Target 'j': prob: 0.7969, logit: 19.88
  - Target ' jou': prob: 0.0005, logit: 12.56
  - Target ' new': prob: 0.0000, logit: 8.00
  - Target 'new': prob: 0.0240, logit: 16.38
  - Target 'Newton': prob: 0.0095, logit: 15.44
  - Target 'p': prob: 0.0048, logit: 14.75
  - Target 'vol': prob: 0.0000, logit: 8.19
  - Target ' volts': prob: 0.0000, logit: 1.17

Source baseline prediction on 'Fact: The official SI unit for the force of a moving engine thrust is named "':
  - Target 'J': prob: 0.0000, logit: 9.94
  - Target 'j': prob: 0.0000, logit: 10.06
  - Target ' jou': prob: 0.0000, logit: 5.34
  - Target ' new': prob: 0.0001, logit: 12.00
  - Target 'new': prob: 0.8164, lo

In [14]:
# Final sparse swap: ENERGY graph-positive features -> FORCE run
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_final_train_config.yaml \
  --graph-json outputs/units_final_energy_graph.json \
  --graph-feature-sign positive \
  --target-token "newtons, joules, volts" \
  --positions all \
  --output outputs/units_final_swap_sparse_energy_to_force.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 192.02it/s]
Loaded 158 nodes from graph JSON; extracted features across 7 layers (97 total features)
  Graph feature sign filter: positive (skipped 43 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: 'new' (prob: 0.8164, logit: 20.62)
  - Target 'j': prob: 0.0000, logit: 10.06
  - Target 'J': prob: 0.0000, logit: 9.94
  - Target ' jou': prob: 0.0000, logit: 5.34
  - Target 'new': prob: 0.8164, logit: 20.62
  - Target ' new': prob: 0.0001, logit: 12.00
  - Target 'Newton': prob: 0.1108, logit: 18.62
  - Target 'p': prob: 0.0280, logit: 17.25
  - Target 'vol': prob: 0.0000, logit: 7.09
  - Target ' volts': prob: 0.0000, logit: -0.35

Source baseline prediction on 'Fact: The official SI unit for the energy of a moving engine thrust is named "':
  - Target 'j': prob: 0.7969, logit: 19.88
  - Target

In [15]:
# Final sparse swap: FORCE graph-positive features -> ENERGY run
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_final_train_config.yaml \
  --graph-json outputs/units_final_force_graph.json \
  --graph-feature-sign positive \
  --target-token "newtons, joules, volts" \
  --positions all \
  --output outputs/units_final_swap_sparse_force_to_energy.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 191.51it/s]
Loaded 158 nodes from graph JSON; extracted features across 7 layers (76 total features)
  Graph feature sign filter: positive (skipped 64 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: 'j' (prob: 0.7969, logit: 19.88)
  - Target 'j': prob: 0.7969, logit: 19.88
  - Target 'J': prob: 0.1226, logit: 18.00
  - Target ' jou': prob: 0.0005, logit: 12.56
  - Target 'new': prob: 0.0240, logit: 16.38
  - Target ' new': prob: 0.0000, logit: 8.00
  - Target 'Newton': prob: 0.0095, logit: 15.44
  - Target 'p': prob: 0.0048, logit: 14.75
  - Target 'vol': prob: 0.0000, logit: 8.19
  - Target ' volts': prob: 0.0000, logit: 1.17

Source baseline prediction on 'Fact: The official SI unit for the force of a moving engine thrust is named "':
  - Target 'j': prob: 0.0000, logit: 10.06
  - Target 'J

In [16]:
# Final sparse inhibition: remove force-graph positive features from FORCE run
!python src/intervention.py \
  --mode inhibit \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --target-token "newtons, joules, volts" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_final_train_config.yaml \
  --graph-json outputs/units_final_force_graph.json \
  --graph-feature-sign positive \
  --positions last \
  --output outputs/units_final_inhibit_force.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 192.21it/s]
Loaded 158 nodes from graph JSON; extracted features across 7 layers (76 total features)
  Graph feature sign filter: positive (skipped 64 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: 'new' (prob: 0.8164, logit: 20.62)
  - Target 'j': prob: 0.0000, logit: 10.06
  - Target ' jou': prob: 0.0000, logit: 5.34
  - Target ' new': prob: 0.0001, logit: 12.00
  - Target 'new': prob: 0.8164, logit: 20.62
  - Target 'Newton': prob: 0.1108, logit: 18.62
  - Target 'p': prob: 0.0280, logit: 17.25
  - Target 'vol': prob: 0.0000, logit: 7.09
  - Target ' volts': prob: 0.0000, logit: -0.35

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition at positions [16]: {}...
Editing token positions:
  16: id=330      token=' "'
  - Top predictio

In [10]:
# Restore the final units SAE files required by heldout_validation.py.
# The validation command starts a fresh Python process and loads these files itself.
from pathlib import Path
import shutil

restore_layers = globals().get('FINAL_LAYERS', [4, 8, 12, 16, 20, 24, 28])
local_sae_dir = Path(globals().get(
    'FINAL_SAE_DIR',
    'mechanistic_data/sae_checkpoints_units_final_train',
))
drive_sae_candidates = [
    Path('/content/drive/MyDrive/mphil-project/mechanistic_data/finals/sae_checkpoints_units_final_train'),
    Path('/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_units_final_train'),
]
required_names = [
    name
    for layer in restore_layers
    for name in (f'sae_layer{layer}.pt', f'sae_layer{layer}_metadata.json')
]
drive_sae_dir = next(
    (path for path in drive_sae_candidates if all((path / name).exists() for name in required_names)),
    None,
)
if drive_sae_dir is None:
    searched = '\n'.join(str(path) for path in drive_sae_candidates)
    raise FileNotFoundError('No complete final units SAE set found in Drive. Searched:\n' + searched)

local_sae_dir.mkdir(parents=True, exist_ok=True)
for name in required_names:
    shutil.copy2(drive_sae_dir / name, local_sae_dir / name)

missing_local = [name for name in required_names if not (local_sae_dir / name).exists()]
if missing_local:
    raise FileNotFoundError('Restore failed for: ' + ', '.join(missing_local))

# Restore the graph too when this tail section is run in a fresh Colab runtime.
local_graph = Path('outputs/units_final_force_graph.json')
drive_graph = Path('/content/drive/MyDrive/mphil-project/outputs/units_final_force_graph.json')
if not local_graph.exists():
    if not drive_graph.exists():
        raise FileNotFoundError(f'Missing units graph locally and in Drive: {drive_graph}')
    local_graph.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_graph, local_graph)

print(f'Restored {len(required_names)} SAE checkpoint/metadata files from {drive_sae_dir}')
print('Local SAE directory:', local_sae_dir.resolve())
print('Units graph:', local_graph.resolve())
print('Latent .npy files are not needed for intervention inference.')

Restored 14 SAE checkpoint/metadata files from /content/drive/MyDrive/mphil-project/mechanistic_data/finals/sae_checkpoints_units_final_train
Local SAE directory: /content/test_run/mechanistic_data/sae_checkpoints_units_final_train
Units graph: /content/test_run/outputs/units_final_force_graph.json
Latent .npy files are not needed for intervention inference.


In [11]:
!python -m src.heldout_validation \
  --units-sae-config configs/sae_units_final_train_config.yaml \
  --units-graph outputs/units_final_force_graph.json \
  --unit-cases 12 \
  --skip-math \
  --positions last \
  --output outputs/units_final_heldout_last.json

Loading language model once for both benchmarks...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 187.89it/s]
Loaded 76 positive units graph features
[units 01/12] bullet impact
[units 02/12] bow string tension
[units 03/12] magnetic repulsion
[units 04/12] spring stretch
[units 05/12] wind gust
[units 06/12] elastic band recoil
[units 07/12] gravity pull
[units 08/12] earthquake tremor
[units 09/12] friction drag
[units 10/12] rocket engine
[units 11/12] water jet
[units 12/12] crane lift

SI-unit generalisation: 6/12 baseline-qualified cases
  full_latent_swap         mean delta=+1.500 95% CI [+1.031, +2.094]  direction=1.00  top-transfer=0.00
  raw_mlp_swap             mean delta=+1.625 95% CI [+1.167, +2.156]  direction=1.00  top-transfer=0.00
  sparse_feature_swap      mean delta=+0.229 95% CI [+0.146, +0.333]  direction=1.00  top-transfer=0.00

Saved held-out valida

## Step 10 - Persist final-train outputs to Drive


In [12]:
# Copy final-train graphs/results to Drive
import os, shutil, glob

drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/test_run/outputs/units_final_*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')


Copied units_final_energy_graph.html
Copied units_final_energy_graph.md
Copied units_final_heldout_last.json
Copied units_final_force_graph.html
Copied units_final_swap_sparse_force_to_energy.json
Copied units_final_swap_sparse_energy_to_force.json
Copied units_final_swap_minpair_force_to_energy.json
Copied units_final_inhibit_force.json
Copied units_final_swap_minpair_energy_to_force.json
Copied units_final_force_graph.md
Copied units_final_force_graph.json
Copied units_final_energy_graph.json
Done!
